In [2]:
import numpy as np

# ----------------------------------------------------
# كل ملف .npz هو "قاموس" (dictionary) مضغوط، بيحتوي مصفوفة أو أكتر
# لازم نفتحه بـ np.load() ونشوف شو المفاتيح (keys) الموجودة فيه
files = {
    "Normal": "normal.npz",
    "Inner Race Fault": "inner_race_fault.npz",
    "Ball Fault": "ball_fault.npz",
    "Outer Race Fault": "outer_race_fault.npz"
}

data = {}  # قاموس رح نخزن فيه كل الإشارات بعد التحميل

for label, filename in files.items():
    npz_file = np.load(filename)
    
    print(f"--- {label} ({filename}) ---")
    print("المفاتيح الموجودة بالملف:", list(npz_file.keys()))
    
    # نخزن إشارة DE (Drive End) تحديدًا - أقرب حساس لموقع المحمل الفعلي
    signal = npz_file['DE']
    data[label] = signal
    
    print("شكل الإشارة (Shape):", signal.shape)
    print("أول 10 قيم:", signal[:10])
    print("نوع البيانات:", signal.dtype)
    print()

--- Normal (normal.npz) ---
المفاتيح الموجودة بالملف: ['DE', 'FE']
شكل الإشارة (Shape): (243938, 1)
أول 10 قيم: [[ 0.05319692]
 [ 0.08866154]
 [ 0.09971815]
 [ 0.05862092]
 [-0.00458954]
 [-0.056952  ]
 [-0.07176369]
 [-0.05862092]
 [-0.04652123]
 [-0.04985908]]
نوع البيانات: float64

--- Inner Race Fault (inner_race_fault.npz) ---
المفاتيح الموجودة بالملف: ['DE', 'FE', 'BA']
شكل الإشارة (Shape): (121265, 1)
أول 10 قيم: [[-0.08300435]
 [-0.19573433]
 [ 0.23341928]
 [ 0.10395848]
 [-0.18111517]
 [ 0.05555281]
 [ 0.17380559]
 [-0.04694375]
 [-0.1119178 ]
 [ 0.05961369]]
نوع البيانات: float64

--- Ball Fault (ball_fault.npz) ---
المفاتيح الموجودة بالملف: ['DE', 'FE', 'BA']
شكل الإشارة (Shape): (122571, 1)
أول 10 قيم: [[-0.0027614 ]
 [-0.09632403]
 [ 0.11370459]
 [ 0.25729725]
 [-0.05831421]
 [-0.12604966]
 [ 0.20742966]
 [ 0.17266854]
 [-0.21993717]
 [-0.15610016]]
نوع البيانات: float64

--- Outer Race Fault (outer_race_fault.npz) ---
المفاتيح الموجودة بالملف: ['DE', 'FE', 'BA']
شكل الإشا

In [3]:
import numpy as np

files = {
    "Normal": "normal.npz",
    "Inner Race Fault": "inner_race_fault.npz",
    "Ball Fault": "ball_fault.npz",
    "Outer Race Fault": "outer_race_fault.npz"
}

data = {}

for label, filename in files.items():
    npz_file = np.load(filename)
    
    # ----------------------------------------------------
    # .flatten() بيحول المصفوفة من شكل (N, 1) لشكل (N,) بسيط
    # يعني من "عمود" لـ "قائمة أرقام متتالية" - أسهل بكتير للتعامل معها لاحقًا
    signal = npz_file['DE'].flatten()
    data[label] = signal
    
    print(f"{label}: شكل بعد التسطيح = {signal.shape}, أول 5 قيم = {signal[:5]}")

# ----------------------------------------------------
# نحسب مدة كل إشارة بالثواني - معدل التسجيل الأصلي لداتاسيت CWRU هو 12,000 عينة/ثانية
sampling_rate = 12000  # Hz

print("\n--- مدة كل إشارة بالثواني ---")
for label, signal in data.items():
    duration = len(signal) / sampling_rate
    print(f"{label}: {duration:.2f} ثانية ({len(signal)} عينة)")

Normal: شكل بعد التسطيح = (243938,), أول 5 قيم = [ 0.05319692  0.08866154  0.09971815  0.05862092 -0.00458954]
Inner Race Fault: شكل بعد التسطيح = (121265,), أول 5 قيم = [-0.08300435 -0.19573433  0.23341928  0.10395848 -0.18111517]
Ball Fault: شكل بعد التسطيح = (122571,), أول 5 قيم = [-0.0027614  -0.09632403  0.11370459  0.25729725 -0.05831421]
Outer Race Fault: شكل بعد التسطيح = (121991,), أول 5 قيم = [ 0.00852784  0.4235496   0.01299481 -0.26517535  0.23715529]

--- مدة كل إشارة بالثواني ---
Normal: 20.33 ثانية (243938 عينة)
Inner Race Fault: 10.11 ثانية (121265 عينة)
Ball Fault: 10.21 ثانية (122571 عينة)
Outer Race Fault: 10.17 ثانية (121991 عينة)


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------
# منرسم بس أول 0.05 ثانية من كل إشارة (600 عينة بمعدل 12kHz) - عشان نشوف التفاصيل بوضوح
# لو رسمنا الإشارة كاملة (10-20 ثانية)، هيك كمية نقاط كبيرة رح "تتلخبط" وما نميز شي
samples_to_plot = 600

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True, sharey=True)

for ax, (label, signal) in zip(axes, data.items()):
    ax.plot(signal[:samples_to_plot])
    ax.set_title(label)
    ax.set_ylabel("Acceleration (g)")

axes[-1].set_xlabel("Sample Index")
plt.tight_layout()
plt.savefig('raw_signals_comparison.png', dpi=110)
print("تم حفظ الرسمة")

تم حفظ الرسمة


In [5]:

WINDOW_SIZE = 1024

def create_windows(signal, window_size):
    """
    تقسم إشارة طويلة لعدة نوافذ متساوية الطول، بدون تداخل (non-overlapping)
    """
    # ----------------------------------------------------
    # نحسب كم نافذة كاملة نقدر نستخرج من الإشارة
    # // هي القسمة الصحيحة (Integer Division) - بترجع بس الجزء الصحيح بدون كسور
    n_windows = len(signal) // window_size
    
    windows = []
    for i in range(n_windows):
        start = i * window_size
        end = start + window_size
        windows.append(signal[start:end])
    
    return np.array(windows)

# ----------------------------------------------------
# نطبق التقسيم على كل الإشارات الأربعة، ونبني قائمتين متوازيتين:
# X_raw: كل النوافذ (لسه إشارات خام، لسه ما استخرجنا خصائص منها)
# y: التصنيف (label) المقابل لكل نافذة
X_raw = []
y = []

for label, signal in data.items():
    windows = create_windows(signal, WINDOW_SIZE)
    X_raw.extend(windows)
    y.extend([label] * len(windows))
    print(f"{label}: {len(windows)} نافذة")

X_raw = np.array(X_raw)
y = np.array(y)

print(f"\nإجمالي عدد النوافذ (العينات): {X_raw.shape[0]}")
print(f"شكل X_raw الكامل: {X_raw.shape}")

# نتأكد من توزيع الفئات
import pandas as pd
print("\nتوزيع الفئات:")
print(pd.Series(y).value_counts())

Normal: 238 نافذة
Inner Race Fault: 118 نافذة
Ball Fault: 119 نافذة
Outer Race Fault: 119 نافذة

إجمالي عدد النوافذ (العينات): 594
شكل X_raw الكامل: (594, 1024)

توزيع الفئات:
Normal              238
Ball Fault          119
Outer Race Fault    119
Inner Race Fault    118
Name: count, dtype: int64


In [6]:

from scipy import stats

def extract_time_features(window):
    """
    تستخرج 6 خصائص إحصائية من نافذة إشارة واحدة (1024 عينة)
    """
    features = {}
    
    features['rms'] = np.sqrt(np.mean(window**2))
    features['peak'] = np.max(np.abs(window))
    features['kurtosis'] = stats.kurtosis(window)
    features['skewness'] = stats.skew(window)
    features['crest_factor'] = features['peak'] / features['rms']
    features['std'] = np.std(window)
    
    return features

# ----------------------------------------------------
# نطبق الدالة على كل نافذة بـ X_raw (594 نافذة)، ونبني جدول Features
feature_list = []

for window in X_raw:
    feats = extract_time_features(window)
    feature_list.append(feats)

# نحول القائمة (list of dictionaries) لجدول pandas مباشرة
features_df = pd.DataFrame(feature_list)
features_df['label'] = y   # نضيف عمود التصنيف

print("شكل جدول الخصائص:", features_df.shape)
print("\n--- أول 5 صفوف ---")
print(features_df.head())

print("\n--- متوسط كل خاصية لكل فئة (هون بنشوف الفروقات!) ---")
print(features_df.groupby('label').mean().round(4))

شكل جدول الخصائص: (594, 7)

--- أول 5 صفوف ---
        rms      peak  kurtosis  skewness  crest_factor       std   label
0  0.075982  0.217794 -0.243664 -0.162976      2.866393  0.074960  Normal
1  0.070425  0.272869  0.184817 -0.107386      3.874580  0.069465  Normal
2  0.078377  0.212370 -0.376182  0.050329      2.709618  0.076996  Normal
3  0.079688  0.226765 -0.034564 -0.163588      2.845658  0.078934  Normal
4  0.076104  0.188588 -0.321609 -0.028448      2.478048  0.074922  Normal

--- متوسط كل خاصية لكل فئة (هون بنشوف الفروقات!) ---
                     rms    peak  kurtosis  skewness  crest_factor     std
label                                                                     
Ball Fault        0.1390  0.4631   -0.0499   -0.0086        3.3328  0.1384
Inner Race Fault  0.2914  1.4117    2.3916    0.1638        4.8453  0.2911
Normal            0.0737  0.2224   -0.2523   -0.0350        3.0193  0.0726
Outer Race Fault  0.6689  3.1753    4.6221    0.0573        4.7509  0.6685


In [7]:

SAMPLING_RATE = 12000  # Hz - معدل التسجيل الأصلي لداتاسيت CWRU

def extract_freq_features(window, fs=SAMPLING_RATE):
    """
    تستخرج خصائص من المجال الترددي لنافذة إشارة واحدة (1024 عينة)
    """
    n = len(window)
    
    # ----------------------------------------------------
    # الخطوة 1: تطبيق FFT - يحول الإشارة من مجال الزمن لمجال التردد
    fft_vals = np.fft.fft(window)
    
    # ----------------------------------------------------
    # الخطوة 2: نحسب "الطيف الترددي" (Magnitude Spectrum)
    # FFT بترجع أرقام عقدية (Complex Numbers)، ونحن مهتمين بـ"القوة" بس مش الطور (Phase)
    magnitude = np.abs(fft_vals)
    
    # ----------------------------------------------------
    # الخطوة 3: نحسب محور التردد المقابل (أي تردد يمثله كل عنصر بمصفوفة FFT)
    freqs = np.fft.fftfreq(n, d=1/fs)
    
    # ----------------------------------------------------
    # الخطوة 4: FFT بيرجع طيف "متماثل" (نصفه الثاني تكرار مرآة للأول)
    # منستخدم بس النصف الأول (الترددات الموجبة) - هو يحتوي كل المعلومة الفعلية
    half_n = n // 2
    magnitude = magnitude[:half_n]
    freqs = freqs[:half_n]
    
    features = {}
    
    # ----------------------------------------------------
    # Spectral Energy: مجموع طاقة كل الترددات - "القوة الكلية" بمجال التردد
    features['spectral_energy'] = np.sum(magnitude**2)
    
    # ----------------------------------------------------
    # Dominant Frequency: أقوى تردد موجود بالإشارة (أين "تتركز" الطاقة أكتر)
    dominant_idx = np.argmax(magnitude)
    features['dominant_freq'] = freqs[dominant_idx]
    features['dominant_freq_magnitude'] = magnitude[dominant_idx]
    
    # ----------------------------------------------------
    # Spectral Centroid: "مركز ثقل" الطيف - متوسط مرجّح بالطاقة لكل الترددات
    features['spectral_centroid'] = np.sum(freqs * magnitude) / np.sum(magnitude)
    
    return features

# نجرب الدالة على أول نافذة
sample_window = X_raw[0]
print(extract_freq_features(sample_window))

{'spectral_energy': np.float64(3107.7448697119767), 'dominant_freq': np.float64(1066.40625), 'dominant_freq_magnitude': np.float64(24.433309743768994), 'spectral_centroid': np.float64(1116.0054752512501)}


In [8]:
# ----------------------------------------------------
# نطبق extract_freq_features على كل نافذة، بنفس منطق الخطوة السابقة
freq_feature_list = []

for window in X_raw:
    feats = extract_freq_features(window)
    freq_feature_list.append(feats)

freq_features_df = pd.DataFrame(freq_feature_list)

# ----------------------------------------------------
# ندمج جدول Time-Domain (features_df من قبل) مع جدول Frequency-Domain الجديد
# pd.concat بيلصق عمودين جنب بعض (axis=1) بناءً على نفس ترتيب الصفوف
final_features_df = pd.concat([features_df, freq_features_df], axis=1)

print("شكل الجدول النهائي:", final_features_df.shape)
print("\n--- أول 5 صفوف ---")
print(final_features_df.head())

print("\n--- متوسط كل خاصية لكل فئة ---")
print(final_features_df.groupby('label').mean().round(3))

شكل الجدول النهائي: (594, 11)

--- أول 5 صفوف ---
        rms      peak  kurtosis  skewness  crest_factor       std   label  \
0  0.075982  0.217794 -0.243664 -0.162976      2.866393  0.074960  Normal   
1  0.070425  0.272869  0.184817 -0.107386      3.874580  0.069465  Normal   
2  0.078377  0.212370 -0.376182  0.050329      2.709618  0.076996  Normal   
3  0.079688  0.226765 -0.034564 -0.163588      2.845658  0.078934  Normal   
4  0.076104  0.188588 -0.321609 -0.028448      2.478048  0.074922  Normal   

   spectral_energy  dominant_freq  dominant_freq_magnitude  spectral_centroid  
0      3107.744870     1066.40625                24.433310        1116.005475  
1      2670.738554     1031.25000                22.848620        1229.160761  
2      3333.057594     1031.25000                26.439782        1145.171336  
3      3392.040660     1031.25000                23.521940        1034.336760  
4      3130.107944     1031.25000                25.338983        1132.525646  

--- مت

In [9]:
# نطبع بس صف groupby لوحده، وبتنسيق أوسع يمنع القطع
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
print(final_features_df.groupby('label').mean().round(3))

                    rms   peak  kurtosis  skewness  crest_factor    std  spectral_energy  dominant_freq  dominant_freq_magnitude  spectral_centroid
label                                                                                                                                              
Ball Fault        0.139  0.463    -0.050    -0.009         3.333  0.138        10243.768       3328.814                   29.001           2654.490
Inner Race Fault  0.291  1.412     2.392     0.164         4.845  0.291        44629.843       3524.960                   62.863           2658.137
Normal            0.074  0.222    -0.252    -0.035         3.019  0.073         2937.073       1044.889                   24.050           1119.923
Outer Race Fault  0.669  3.175     4.622     0.057         4.751  0.668       235234.843       3425.322                  156.490           3169.089


In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------
# نرسم 4 رسومات Boxplot جنب بعض - وحدة لكل خاصية قوية
top_features = ['rms', 'kurtosis', 'spectral_energy', 'peak']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, feature in zip(axes, top_features):
    # .boxplot() من pandas مباشرة - بيرسم صندوق منفصل لكل قيمة فريدة بعمود label
    final_features_df.boxplot(column=feature, by='label', ax=ax)
    ax.set_title(feature)
    ax.set_xlabel('')
    plt.setp(ax.get_xticklabels(), rotation=20)

plt.suptitle('')  # نشيل العنوان التلقائي الزائد اللي pandas بيضيفه
plt.tight_layout()
plt.savefig('features_boxplot.png', dpi=110)
print("تم حفظ الرسمة")

تم حفظ الرسمة


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

# ----------------------------------------------------
# X: كل الأعمدة الرقمية (9 خصائص)، y: عمود label
X = final_features_df.drop('label', axis=1)
y = final_features_df['label']

print("شكل X:", X.shape)
print("شكل y:", y.shape)

# ----------------------------------------------------
# تقسيم عشوائي مع stratify - كل صف (نافذة) مستقل، فالتقسيم العشوائي مناسب هون
# (بعكس مشروع الـ Churn يلي احتاج تقسيم زمني بمرحلة بناء RFM)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nتوزيع y_train:")
print(y_train.value_counts())

# ----------------------------------------------------
# نجرب عدة موديلات، زي منهجية مشروع الـ Churn بالضبط
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "SVM": SVC(random_state=42)
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    results.append({"Model": name, "Accuracy": acc, "F1 (macro)": f1_macro})
    
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values("F1 (macro)", ascending=False)
print("\n\n--- جدول المقارنة النهائي ---")
print(results_df.to_string(index=False))

شكل X: (594, 10)
شكل y: (594,)

توزيع y_train:
label
Normal              190
Ball Fault           95
Outer Race Fault     95
Inner Race Fault     95
Name: count, dtype: int64

=== Logistic Regression ===
                  precision    recall  f1-score   support

      Ball Fault       1.00      1.00      1.00        24
Inner Race Fault       1.00      1.00      1.00        23
          Normal       1.00      1.00      1.00        48
Outer Race Fault       1.00      1.00      1.00        24

        accuracy                           1.00       119
       macro avg       1.00      1.00      1.00       119
    weighted avg       1.00      1.00      1.00       119


=== Decision Tree ===
                  precision    recall  f1-score   support

      Ball Fault       1.00      1.00      1.00        24
Inner Race Fault       1.00      1.00      1.00        23
          Normal       1.00      1.00      1.00        48
Outer Race Fault       1.00      1.00      1.00        24

        accura

In [12]:
# ----------------------------------------------------
# نبني تقسيم Train/Test زمني: لكل فئة لحالها، ناخذ أول 80% من نوافذها كـ Train
# وآخر 20% (زمنيًا، متتالية) كـ Test - بدون أي خلط عشوائي
X_train_list, X_test_list, y_train_list, y_test_list = [], [], [], []

# نحتاج نعرف مين النوافذ تبع كل فئة بالترتيب الأصلي (زمنيًا)
final_features_df['original_index'] = range(len(final_features_df))

for label in final_features_df['label'].unique():
    class_data = final_features_df[final_features_df['label'] == label].sort_values('original_index')
    split_point = int(len(class_data) * 0.8)
    
    X_train_list.append(class_data.iloc[:split_point])
    X_test_list.append(class_data.iloc[split_point:])

train_df = pd.concat(X_train_list)
test_df = pd.concat(X_test_list)

X_train_t = train_df.drop(['label', 'original_index'], axis=1)
y_train_t = train_df['label']
X_test_t = test_df.drop(['label', 'original_index'], axis=1)
y_test_t = test_df['label']

print("حجم Train:", X_train_t.shape, " حجم Test:", X_test_t.shape)

# نعيد تجربة نفس الموديلات على هاد التقسيم "الأصعب والأصدق"
for name, model in models.items():
    pipeline = Pipeline([("scaler", StandardScaler()), ("model", model)])
    pipeline.fit(X_train_t, y_train_t)
    y_pred_t = pipeline.predict(X_test_t)
    print(f"\n=== {name} (Time-based Split) ===")
    print(classification_report(y_test_t, y_pred_t, zero_division=0))

حجم Train: (474, 10)  حجم Test: (120, 10)

=== Logistic Regression (Time-based Split) ===
                  precision    recall  f1-score   support

      Ball Fault       1.00      1.00      1.00        24
Inner Race Fault       1.00      1.00      1.00        24
          Normal       1.00      1.00      1.00        48
Outer Race Fault       1.00      1.00      1.00        24

        accuracy                           1.00       120
       macro avg       1.00      1.00      1.00       120
    weighted avg       1.00      1.00      1.00       120


=== Decision Tree (Time-based Split) ===
                  precision    recall  f1-score   support

      Ball Fault       1.00      1.00      1.00        24
Inner Race Fault       1.00      1.00      1.00        24
          Normal       1.00      1.00      1.00        48
Outer Race Fault       1.00      1.00      1.00        24

        accuracy                           1.00       120
       macro avg       1.00      1.00      1.00    

In [13]:
import joblib

# ----------------------------------------------------
# ندرب الموديل النهائي على كل البيانات المتاحة (Train+Test) - لأنه هلق خلصنا التقييم
# وبدنا "أفضل موديل ممكن" للاستخدام الحقيقي، مش محتاجين نحتفظ بـTest بعد الآن
final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=1000))
])

final_pipeline.fit(X, y)   # X, y هما كل الـ594 عينة (قبل أي تقسيم)

# نحفظ الـ Pipeline كاملة (Scaler + Model مع بعض) بملف واحد
joblib.dump(final_pipeline, "bearing_diagnosis_model.pkl")
print("تم حفظ الموديل بنجاح")

تم حفظ الموديل بنجاح


In [14]:
import numpy as np
from scipy import stats

def diagnose_bearing(raw_signal_window, model_path="bearing_diagnosis_model.pkl", fs=12000):
    """
    دالة كاملة تحاكي كيف رح تشتغل على متحكم حقيقي:
    تاخد إشارة اهتزاز خام (1024 عينة)، وترجع التشخيص + مستوى الثقة
    
    راح تكون هاي الدالة بالضبط اللي تنادى بحلقة القراءة المستمرة من الحساس (Sensor Loop)
    على ESP32/Raspberry Pi، كل ما توصل نافذة جديدة من قراءات الحساس
    """
    # ----------------------------------------------------
    # خطوة 0: التحقق من صحة المُدخل (ضروري جدًا بأنظمة حقيقية - حساسات ممكن تعطي بيانات فاسدة)
    if len(raw_signal_window) != 1024:
        raise ValueError(f"الإشارة لازم تكون 1024 عينة بالضبط، وصلت {len(raw_signal_window)}")
    
    raw_signal_window = np.array(raw_signal_window)
    
    # ----------------------------------------------------
    # خطوة 1: استخراج خصائص Time-Domain (نفس الدالة يلي بنيناها قبل، بدون أي تغيير)
    time_feats = extract_time_features(raw_signal_window)
    
    # ----------------------------------------------------
    # خطوة 2: استخراج خصائص Frequency-Domain (نفس الدالة، بدون أي تغيير)
    freq_feats = extract_freq_features(raw_signal_window, fs=fs)
    
    # ----------------------------------------------------
    # خطوة 3: دمج كل الخصائص بنفس الترتيب بالضبط يلي تدرب عليه الموديل
    all_feats = {**time_feats, **freq_feats}
    feature_vector = pd.DataFrame([all_feats])[X.columns]  # X.columns يضمن نفس ترتيب الأعمدة الأصلي بالضبط
    
    # ----------------------------------------------------
    # خطوة 4: تحميل الموديل المحفوظ (الـ Scaler + الموديل مع بعض) والتنبؤ
    pipeline = joblib.load(model_path)
    prediction = pipeline.predict(feature_vector)[0]
    probabilities = pipeline.predict_proba(feature_vector)[0]
    
    # نبني قاموس بالثقة لكل فئة ممكنة - مهم جدًا عمليًا، مش بس "القرار النهائي"
    confidence = dict(zip(pipeline.classes_, probabilities.round(3)))
    
    return {
        "diagnosis": prediction,
        "confidence": confidence[prediction],
        "all_probabilities": confidence
    }

In [15]:
np.random.seed(123)  # عشان نتائج قابلة للتكرار

# ----------------------------------------------------
# إشارة 1: محاكاة "محمل سليم" - ضجيج عشوائي بسيط بسعة منخفضة (زي Normal الحقيقية)
synthetic_normal = np.random.normal(loc=0, scale=0.08, size=1024)

# ----------------------------------------------------
# إشارة 2: محاكاة "عطل بالحلقة الخارجية" - ضجيج خلفي + نبضات دورية منتظمة قوية
# نبني نبضة على شكل "طقة" مخمّدة (decaying impulse)، ونكررها بفواصل منتظمة (محاكاة BPFO)
t = np.arange(1024)
synthetic_outer_fault = np.random.normal(loc=0, scale=0.1, size=1024)  # ضجيج خلفي

impulse_period = 110    # فاصل زمني بين النبضات (بالعينات) - مشابه لما شفناه بالرسمة الحقيقية
impulse_width = 15

for start in range(0, 1024, impulse_period):
    for i in range(impulse_width):
        if start + i < 1024:
            # نبضة مخمّدة أسيًا (exponentially decaying) - محاكاة صدمة ميكانيكية حقيقية
            synthetic_outer_fault[start + i] += 2.5 * np.exp(-i / 3) * np.sin(2 * np.pi * i / 3)

# ----------------------------------------------------
# نختبر الدالة على الإشارتين المصطنعتين
print("=== اختبار إشارة مصطنعة سليمة ===")
result_normal = diagnose_bearing(synthetic_normal)
print(result_normal)

print("\n=== اختبار إشارة مصطنعة بعطل Outer Race ===")
result_fault = diagnose_bearing(synthetic_outer_fault)
print(result_fault)

=== اختبار إشارة مصطنعة سليمة ===
{'diagnosis': 'Ball Fault', 'confidence': np.float64(0.998), 'all_probabilities': {'Ball Fault': np.float64(0.998), 'Inner Race Fault': np.float64(0.001), 'Normal': np.float64(0.0), 'Outer Race Fault': np.float64(0.0)}}

=== اختبار إشارة مصطنعة بعطل Outer Race ===
{'diagnosis': 'Inner Race Fault', 'confidence': np.float64(1.0), 'all_probabilities': {'Ball Fault': np.float64(0.0), 'Inner Race Fault': np.float64(1.0), 'Normal': np.float64(0.0), 'Outer Race Fault': np.float64(0.0)}}


In [16]:
# ----------------------------------------------------
# الخطوة 1: نستخرج "التردد المهيمن الحقيقي" لكل فئة عطل من الداتا الأصلية
# بدل ما نخمن impulse_period، نحسبه من نافذة حقيقية فعلية
outer_fault_real_window = X_raw[y == 'Outer Race Fault'][0]  # أول نافذة حقيقية من Outer Race Fault
real_dominant_freq = extract_freq_features(outer_fault_real_window)['dominant_freq']

print(f"التردد المهيمن الحقيقي لـ Outer Race Fault: {real_dominant_freq:.1f} Hz")

# نحول التردد لفاصل زمني بالعينات (Period = fs / frequency)
real_impulse_period = int(12000 / real_dominant_freq)
print(f"الفاصل الزمني المقابل (بالعينات): {real_impulse_period}")

# ----------------------------------------------------
# الخطوة 2: نعيد بناء الإشارة المصطنعة بهاد الفاصل الزمني "المُعاير" الصحيح
synthetic_outer_fault_v2 = np.random.normal(loc=0, scale=0.1, size=1024)

impulse_width = 15
for start in range(0, 1024, real_impulse_period):
    for i in range(impulse_width):
        if start + i < 1024:
            synthetic_outer_fault_v2[start + i] += 2.5 * np.exp(-i / 3) * np.sin(2 * np.pi * i / 3)

print("\n=== إعادة اختبار Outer Race Fault (بعد التصحيح) ===")
result_v2 = diagnose_bearing(synthetic_outer_fault_v2)
print(result_v2)

التردد المهيمن الحقيقي لـ Outer Race Fault: 3445.3 Hz
الفاصل الزمني المقابل (بالعينات): 3

=== إعادة اختبار Outer Race Fault (بعد التصحيح) ===
{'diagnosis': 'Outer Race Fault', 'confidence': np.float64(1.0), 'all_probabilities': {'Ball Fault': np.float64(0.0), 'Inner Race Fault': np.float64(0.0), 'Normal': np.float64(0.0), 'Outer Race Fault': np.float64(1.0)}}


In [18]:
# ----------------------------------------------------
# نفس المنهجية بالضبط: نستخرج المعلومات الفيزيائية من نافذة حقيقية أولاً
inner_fault_real_window = X_raw[y == 'Inner Race Fault'][0]
real_dominant_freq_inner = extract_freq_features(inner_fault_real_window)['dominant_freq']
real_kurtosis_inner = extract_time_features(inner_fault_real_window)['kurtosis']
real_rms_inner = extract_time_features(inner_fault_real_window)['rms']

print(f"التردد المهيمن الحقيقي لـ Inner Race Fault: {real_dominant_freq_inner:.1f} Hz")
print(f"Kurtosis الحقيقي: {real_kurtosis_inner:.2f}")
print(f"RMS الحقيقي: {real_rms_inner:.3f}")

# ----------------------------------------------------
# نبني إشارة مصطنعة بنفس أسلوب "الرنين داخل نبضات متكررة" (تصحيح الدرس يلي تعلمناه)
# نفصل بوضوح بين: تردد تكرار النبضات (منخفض، من الرسمة البصرية) وتردد الرنين الداخلي (عالي، من FFT)
np.random.seed(456)

impulse_repetition_period = 100   # فاصل تكرار النبضات (تقريبي، بصري - يشبه ما شفناه بـInner Race بالرسمة)
impulse_width = 20

synthetic_inner_fault = np.random.normal(loc=0, scale=real_rms_inner * 0.5, size=1024)

for start in range(0, 1024, impulse_repetition_period):
    for i in range(impulse_width):
        if start + i < 1024:
            # الرنين الداخلي يستخدم التردد الحقيقي المُستخرج من FFT
            ringing_period_samples = 12000 / real_dominant_freq_inner
            synthetic_inner_fault[start + i] += 1.5 * np.exp(-i / 5) * np.sin(2 * np.pi * i / ringing_period_samples)

# ----------------------------------------------------
# نختبر الدالة الكاملة على الإشارة الجديدة
print("\n=== اختبار إشارة مصطنعة بعطل Inner Race ===")
result_inner = diagnose_bearing(synthetic_inner_fault)
print(result_inner)

التردد المهيمن الحقيقي لـ Inner Race Fault: 3585.9 Hz
Kurtosis الحقيقي: 2.25
RMS الحقيقي: 0.293

=== اختبار إشارة مصطنعة بعطل Inner Race ===
{'diagnosis': 'Inner Race Fault', 'confidence': np.float64(1.0), 'all_probabilities': {'Ball Fault': np.float64(0.0), 'Inner Race Fault': np.float64(1.0), 'Normal': np.float64(0.0), 'Outer Race Fault': np.float64(0.0)}}
